<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0_0911.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 주제

행동 특성을 이용한 외향형(Extrovert) / 내향형(Introvert) 성격 분류

- 혼자 보내는 시간, 사회적 행사 참여도, 외출 빈도, 친구 수, SNS 게시 빈도 등의 행동 특성을 이용해 개인의 성격 유형을 예측하는 이진 분류 문제

# 데이터

Kaggle Playgroud Series - Extrovert vs. Introvert

- Train 데이터: 18,524개 샘플

- Target: Personality -> Introvert / Extrovert

- 주요 변수

Time_spent_Alone: 혼자 보내는 시간

Stage_fear: 무대 공포 여부

Social_event_attendance: 사회적 행사 참여도

Going_outside: 외출 빈도

Drained_after_socialing: 사회활동 후 피로 여부

Friends_circle_size: 친구 관계 규모

Post_frequency: SNS 게시 빈도

- 수치형 변수, 범주형 변수 모두 결측치 존재


# 코드 흐름

1. 데이터 로드 & 결측치 처리

- 수치형 변수 -> 중앙값(median)

- 범주형 변수 -> 최빈값(mode)



In [ ]:
for col in num_cols:
  df[col]=df[col].fillna(df[col].median())

for col in cat_cols:
  df[col]=df[col].fillna(df[col].mode()[0])

2. 전처리

- Personality: Introvert=0, Extrovert=1

- 범주형 변수 -> LabelEncoder

- Time_spent_Alone -> PowerTransformer

- 수치형 변수 -> StadardScaler

In [ ]:
df["Personality"]=df["Personality"].map(
    ("Introvert":0, "Extrovert":1}
)

    preprocessor=ColumnTransformer([
        ('time', tsa_p, time),
        ('num', scaler, num_cols)
    ])

3. 모델 학습 및 비교

- Logistic Regression

- Random Forest

- XGBoost

- StratifiedKFold 5-fold CV

- F1 / Precision / Recall 비교

4. 하이퍼파라미터 튜닝

RandomizedSearchCV로 각 모델의 최적 파라미터 탐색.

튜닝 결과 Random Forest는 n_estimators=500, max_dfpth=8 등이 선택됨.

In [ ]:
rs=RandomizedSearchCV(
    full_pip,
    param_distributions=param_grids[name],
    n_iter=10,
    scoring="f1",
    cv=cv
)

5. Stacking Ensemble

튜닝한 logistic Regression, Random Forest, XGBoost를 결합하고 Logistic Regression을 최종 meta model로 사용.

6. 성능 평가

최종 Stacking 모델의 Accuracy=0.964, weighted F1-score도 약 0.96을 기록.

# 새롭게 알게 된 내용 / 어려운 내용 / 배울 점

1. 불균형 데이터 처리 방법

- 단순히 모델을 학습하기보다 class_weight='balanced', scale_pos_weight를 이용하여 클래스 불균형을 반영할 수 있다는 점을 알게 됨.

- StratifiedKFold를 사용하면 각 fold에서도 클래스 비율을 유지하며 검증할 수 있음.

2. Pipeline과 ColumnTransformer 활용

- 변수마다 서로 다른 전처리 방법을 적용하고, 이를 모델 학습 과정과 하나의 Pipelime으로 연결할 수 있다는 점을 배움.

- 특히 수치형 변수의 스케일링과 특정 변수의 분포 변환을 동시에 처리할 수 있음.

3. Stacking Ensemble

- 하나의 모델만 선택하는 것이 아니라 여러 모델의 예측을 결합하고, 그 결과를 다시 meta model이 학습하여 최종 예측을 수행하는 방식이 인상적이었음.

- 개별 모델을 단순 평균하는 것과 Stacking의 차이를 이해하는 데 다소 어려움이 있었음.

-> 단순 평균 / Voting은 평균-> 최종 예측 Stacking은 각 모델의 예측값 + Meta Model -> 최종 예측

4. 하이퍼파라미터 탐색

- 모든 조합을 탐색하는 GridSearchCV와 달리 RandomizedSearchCV는 일부 조합을 무작위로 탐색하여 계산량을 줄일 수 있다는 점을 배움.